# Learning LLM Engineering: Working with Transformers

I am exploring the lower-level API of the `transformers` library to understand how models wrap PyTorch code. My goal is to run these experiments on a T4 GPU.

In [ ]:
# Install necessary libraries for 4-bit quantization and transformer modeling
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [ ]:
# Authenticate with Hugging Face using my stored secret token
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Llama 3.1 is larger and you should already be approved
# see here: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

# LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

In [ ]:
# Configure 4-bit quantization to fit larger models into the T4 GPU memory
# Using NF4 (Normal Float 4) and Double Quantization for better efficiency
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Setup the tokenizer and prepare the chat input for the model
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
# Apply the chat template to convert the dictionary message into model-readable tokens
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [ ]:
inputs

In [ ]:
# The model
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

## Inspecting the Transformer Architecture

I am printing the `model` object to see the underlying PyTorch layers. Here is what I am looking for to build my intuition:

- **Embeddings**: How tokens are mapped to high-dimensional vectors.
- **Decoder Layers**: The core blocks consisting of Self-Attention and MLP layers.
- **LM Head**: The final layer that maps internal representations back to vocabulary probabilities.
- **Quantization**: Verification that layers are loaded in 4-bit.

In [ ]:
# Execute this cell and look at what gets printed; investigate the layers
model

### And if you want to go even deeper into Transformers

In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:  
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:  
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not neceesary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [ ]:
# OK, with that, now let's run the model!
outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

In [ ]:
# Well that doesn't make much sense!
# How about this..
tokenizer.decode(outputs[0])

In [ ]:
# Manually clear objects and empty the CUDA cache to prevent Out-of-Memory (OOM) errors
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

## Implementation Notes: Streaming and Templates

- **TextStreamer**: I'm using this to see the model's output in real-time as it's generated.
- **add_generation_prompt**: Setting this to `True` ensures the model knows it's time to start the assistant's turn, rather than continuing the user's text.

In [ ]:
# Defining a reusable helper function to load and run different models easily
def generate(model_name, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = tokenizer.eos_token

  # Prepare inputs with a generation prompt to trigger the assistant response
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")

  # Setup the live streamer
  streamer = TextStreamer(tokenizer)

  # Load model with or without quantization based on size/memory
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

  # Run inference
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)

In [ ]:
generate(QWEN, messages)

In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)